# O5 — Módulo de visualización

Mapas interactivos que comparan la **ruta real** del ave con la **predicción** del modelo Random Forest entrenado en `ML3.ipynb` sobre `hmm5.csv` (versión canónica).

**Pipeline**
1. Reproducir ML3 — RF con 9 features sobre `hmm5.csv`.
2. Reconstruir centroide geográfico de la celda predicha.
3. Calcular error haversine (km) entre centroide predicho y posición real del día siguiente.
4. Resumir error por estado HMM, mes y animal.
5. Generar mapas Folium por animal y exportar HTML interactivos a `img/o5/`.

## 1. Reentrenar Random Forest (referencia ML3)

Mismas features, mismo split por animal 80/20, mismo `LabelEncoder` ajustado solo en train. La única diferencia respecto a ML3 es que aquí guardamos también `lon`, `lat`, `date`, `animal_id` y `target_cell` para poder reconstruir las trayectorias en el mapa.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv('../data/processed/hmm5.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['animal_id', 'trayectoria_id', 'date']).reset_index(drop=True)

# Grid 0.5° (igual que ML3)
GRID_RES = 0.5
lon_min, lat_min = df['lon'].min(), df['lat'].min()
df['grid_x'] = ((df['lon'] - lon_min) / GRID_RES).astype(int)
df['grid_y'] = ((df['lat'] - lat_min) / GRID_RES).astype(int)
df['cell_id'] = df['grid_x'].astype(str) + '_' + df['grid_y'].astype(str)

# Target = celda del día siguiente (dentro de la misma trayectoria)
df['target_cell'] = df.groupby('trayectoria_id')['cell_id'].shift(-1)
df['next_lat']    = df.groupby('trayectoria_id')['lat'].shift(-1)
df['next_lon']    = df.groupby('trayectoria_id')['lon'].shift(-1)
df['semana_num']  = df['date'].dt.isocalendar().week.astype(int)

df_filt = df.dropna(subset=['target_cell']).copy()
print(f'Filas tras crear target: {len(df_filt):,}')
print(f'Animales: {df_filt["animal_id"].nunique()} | Trayectorias: {df_filt["trayectoria_id"].nunique()}')

Filas tras crear target: 20,601
Animales: 117 | Trayectorias: 480


In [2]:
# Split por animal: primer 80% cronológico → train, último 20% → test
features = ['grid_x', 'grid_y', 'step_length', 'turning_angle', 'bearing',
            'estado_hmm', 'veg_low', 'veg_high', 'semana_num']

train_parts, test_parts = [], []
for animal, g in df_filt.groupby('animal_id'):
    g = g.sort_values('date')
    if len(g) < 5:
        continue
    cut = int(len(g) * 0.8)
    train_parts.append(g.iloc[:cut])
    test_parts.append(g.iloc[cut:])

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df  = pd.concat(test_parts).reset_index(drop=True)

# LabelEncoder ajustado SOLO en train
le = LabelEncoder()
y_train = le.fit_transform(train_df['target_cell'])

# Filtrar test: descartar filas cuya target_cell no exista en train
mask_seen = test_df['target_cell'].isin(le.classes_)
test_df = test_df[mask_seen].reset_index(drop=True)
y_test  = le.transform(test_df['target_cell'])

X_train = train_df[features]
X_test  = test_df[features]

print(f'Train: {len(X_train):,} | Test: {len(X_test):,} | Clases: {len(le.classes_)}')

Train: 16,415 | Test: 3,782 | Clases: 960


In [3]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Random Forest Top-1 sobre test: {acc:.4f} (esperado ≈ 0.8115)')

Random Forest Top-1 sobre test: 0.8115 (esperado ≈ 0.8115)


## 2. Reconstruir coordenadas predichas y calcular error

El modelo predice una `cell_id` (string `"grid_x_grid_y"`). Para colocarla en el mapa convertimos el índice de celda a su **centroide geográfico** invirtiendo la fórmula del grid:

$$\text{lon\_centroide} = \text{lon\_min} + (g_x + 0.5)\cdot \Delta,\qquad \text{lat\_centroide} = \text{lat\_min} + (g_y + 0.5)\cdot \Delta$$

El error en km se calcula con la fórmula **haversine** entre el centroide predicho y la posición real del día siguiente (`next_lat`, `next_lon`).

In [4]:
def cell_to_center(cell_id, lon_min=lon_min, lat_min=lat_min, res=GRID_RES):
    gx, gy = map(int, cell_id.split('_'))
    return lat_min + (gy + 0.5) * res, lon_min + (gx + 0.5) * res

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1r, lat2r = np.radians(lat1), np.radians(lat2)
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

pred_cells = le.inverse_transform(y_pred)
test_df = test_df.copy()
test_df['cell_pred'] = pred_cells
test_df[['lat_pred', 'lon_pred']] = pd.DataFrame(
    [cell_to_center(c) for c in pred_cells], index=test_df.index)

# Centroide de la celda real (next_*) → para comparar el error que aporta SOLO el modelo
# y separarlo del error "intrínseco" de la rejilla.
test_df[['lat_real_centroide', 'lon_real_centroide']] = pd.DataFrame(
    [cell_to_center(c) for c in test_df['target_cell']], index=test_df.index)

test_df['error_km'] = haversine_km(test_df['lat_pred'], test_df['lon_pred'],
                                    test_df['next_lat'], test_df['next_lon'])
test_df['error_celda_km'] = haversine_km(test_df['lat_pred'], test_df['lon_pred'],
                                         test_df['lat_real_centroide'], test_df['lon_real_centroide'])
test_df['acierto'] = test_df['cell_pred'] == test_df['target_cell']
print(f'Test con error calculado: {len(test_df):,}')
test_df[['animal_id', 'date', 'estado_hmm', 'cell_pred', 'target_cell', 'error_km', 'acierto']].head()

Test con error calculado: 3,782


,animal_id,date,estado_hmm,cell_pred,target_cell,error_km,acierto
0,91732A,2010-07-12,1,32_127,36_126,122.775497,False
1,91732A,2010-07-13,1,36_126,36_126,15.891828,True
2,91732A,2010-07-14,0,32_127,33_127,31.756041,False
3,91732A,2010-07-15,1,33_127,33_127,17.629152,True
4,91732A,2010-07-16,1,32_127,32_127,12.843916,True


## 3. Resumen del error

Métricas globales y por estado HMM (0 = migración, 1 = estacionario).

In [5]:
def resumen(df_):
    return pd.Series({
        'n':              len(df_),
        'acierto_top1':   df_['acierto'].mean(),
        'mediana_km':     df_['error_km'].median(),
        'media_km':       df_['error_km'].mean(),
        'p90_km':         df_['error_km'].quantile(0.90),
        'pct_<=50km':     (df_['error_km'] <= 50).mean(),
        'pct_<=100km':    (df_['error_km'] <= 100).mean(),
    })

resumen_global = resumen(test_df).to_frame('GLOBAL').T
resumen_estado = test_df.groupby(test_df['estado_hmm'].map({0: 'migración', 1: 'estacionario'})).apply(resumen)
resumen_total  = pd.concat([resumen_global, resumen_estado])
resumen_total = resumen_total.round({'acierto_top1': 4, 'mediana_km': 2, 'media_km': 2,
                                     'p90_km': 2, 'pct_<=50km': 4, 'pct_<=100km': 4})
resumen_total

,n,acierto_top1,mediana_km,media_km,p90_km,pct_<=50km,pct_<=100km
GLOBAL,3782.0,0.8115,23.89,48.50,40.46,0.9178,0.9466
estacionario,3448.0,0.8643,23.34,39.37,34.15,0.9632,0.9727
migración,334.0,0.2665,56.22,142.78,340.74,0.4491,0.6766


In [6]:
test_df['mes'] = test_df['date'].dt.month
resumen_mes = test_df.groupby('mes').apply(resumen).round(2)
resumen_mes

,n,acierto_top1,mediana_km,media_km,p90_km,pct_<=50km,pct_<=100km
mes,,,,,,,
1,215.0,0.96,25.52,34.17,33.21,0.99,0.99
2,272.0,0.87,25.69,56.00,36.87,0.96,0.97
3,317.0,0.79,27.21,67.04,43.35,0.91,0.94
4,191.0,0.63,27.84,65.15,63.04,0.88,0.92
5,178.0,0.91,19.04,28.06,32.35,0.94,0.97
6,221.0,0.95,18.35,17.61,30.59,1.00,1.00
7,348.0,0.88,20.98,34.29,27.92,0.97,0.99
8,459.0,0.75,22.86,28.57,38.74,0.92,0.97
9,423.0,0.77,22.73,55.56,145.17,0.81,0.88


## 4. Distribución del error con Plotly

In [7]:
import plotly.express as px
import plotly.graph_objects as go

test_df['estado_label'] = test_df['estado_hmm'].map({0: 'migración', 1: 'estacionario'})

fig_hist = px.histogram(
    test_df, x='error_km', color='estado_label', nbins=80,
    color_discrete_map={'migración': '#e74c3c', 'estacionario': '#3498db'},
    title='Distribución del error de predicción (km) por estado HMM',
    labels={'error_km': 'Error (km)', 'estado_label': 'Estado HMM'},
    opacity=0.75
)
fig_hist.update_layout(barmode='overlay', xaxis_range=[0, 600], height=400)
fig_hist.show()
fig_hist.write_html('../img/o5/error_histograma.html', include_plotlyjs='cdn')

In [8]:
fig_cdf = go.Figure()
for estado, color in [('migración', '#e74c3c'), ('estacionario', '#3498db')]:
    err = np.sort(test_df.loc[test_df['estado_label'] == estado, 'error_km'].values)
    cdf = np.arange(1, len(err) + 1) / len(err)
    fig_cdf.add_trace(go.Scatter(x=err, y=cdf, mode='lines', name=estado, line=dict(color=color, width=2)))
fig_cdf.update_layout(
    title='CDF del error de predicción por estado HMM',
    xaxis_title='Error (km)', yaxis_title='Fracción acumulada',
    xaxis_range=[0, 500], height=400
)
fig_cdf.show()
fig_cdf.write_html('../img/o5/error_cdf.html', include_plotlyjs='cdn')

In [9]:
fig_box = px.box(
    test_df, x='mes', y='error_km', color='estado_label',
    color_discrete_map={'migración': '#e74c3c', 'estacionario': '#3498db'},
    title='Error mensual por estado HMM',
    labels={'mes': 'Mes', 'error_km': 'Error (km)', 'estado_label': 'Estado HMM'},
    points=False
)
fig_box.update_layout(yaxis_range=[0, 400], height=400)
fig_box.show()
fig_box.write_html('../img/o5/error_mensual.html', include_plotlyjs='cdn')

## 5. Mapas interactivos por animal con Folium

Para cada animal seleccionado se dibuja:
- **Ruta real** completa (línea negra fina) — train + test.
- **Tramo test real** (línea coloreada por estado HMM): rojo migración, azul estacionario.
- **Predicción** del modelo (línea punteada verde).
- **Marcadores** día a día con tooltip (fecha, estado, celda real, celda predicha, error en km).
- **Líneas de error** (gris) que conectan posición real y predicción cada día — su longitud visualiza el error.

In [10]:
import folium
from folium.plugins import AntPath

def mapa_animal(animal_id, df_real_full, df_test_pred, output_html=None):
    """Mapa Folium ruta real vs. predicha para un animal.

    df_real_full: dataframe completo (df_filt) con todas las posiciones del ave.
    df_test_pred: subset de test_df con predicciones para ese ave.
    """
    real = df_real_full[df_real_full['animal_id'] == animal_id].sort_values('date')
    pred = df_test_pred[df_test_pred['animal_id'] == animal_id].sort_values('date')
    if len(pred) == 0:
        print(f'{animal_id}: sin filas de test')
        return None

    centro = [real['lat'].mean(), real['lon'].mean()]
    m = folium.Map(location=centro, zoom_start=5, tiles='cartodbpositron')

    # Ruta real completa (gris claro, fondo de contexto)
    coords_real_full = real[['lat', 'lon']].values.tolist()
    folium.PolyLine(coords_real_full, color='#888', weight=1.5, opacity=0.7,
                    tooltip=f'Ruta real completa ({len(real)} días)').add_to(m)

    # Tramo test real coloreado por estado
    grupo_real_mig = folium.FeatureGroup(name='Test real — migración', show=True)
    grupo_real_est = folium.FeatureGroup(name='Test real — estacionario', show=True)
    grupo_pred     = folium.FeatureGroup(name='Predicción RF', show=True)
    grupo_lineas   = folium.FeatureGroup(name='Líneas de error', show=True)

    for _, r in pred.iterrows():
        lat_r, lon_r = r['next_lat'], r['next_lon']
        lat_p, lon_p = r['lat_pred'], r['lon_pred']
        color_real   = '#e74c3c' if r['estado_hmm'] == 0 else '#3498db'
        tooltip = (f"<b>{animal_id}</b> — {r['date'].strftime('%Y-%m-%d')}<br>"
                   f"Estado: {'migración' if r['estado_hmm']==0 else 'estacionario'}<br>"
                   f"Celda real: {r['target_cell']}<br>"
                   f"Celda predicha: {r['cell_pred']}<br>"
                   f"Error: {r['error_km']:.1f} km — {'✓ acierto' if r['acierto'] else '✗ fallo'}")

        # Marcador real
        marker_r = folium.CircleMarker([lat_r, lon_r], radius=4,
                                       color=color_real, fill=True, fill_opacity=0.85,
                                       tooltip=tooltip)
        marker_r.add_to(grupo_real_mig if r['estado_hmm'] == 0 else grupo_real_est)

        # Marcador predicho
        folium.CircleMarker([lat_p, lon_p], radius=3, color='#27ae60',
                            fill=True, fill_opacity=0.7, tooltip=tooltip
                           ).add_to(grupo_pred)

        # Línea de error
        folium.PolyLine([[lat_r, lon_r], [lat_p, lon_p]],
                        color='#666', weight=1, opacity=0.5).add_to(grupo_lineas)

    # Polilíneas conectoras
    coords_test_real = pred[['next_lat', 'next_lon']].values.tolist()
    coords_test_pred = pred[['lat_pred', 'lon_pred']].values.tolist()
    folium.PolyLine(coords_test_real, color='#222', weight=2, opacity=0.85,
                    tooltip='Trayectoria real (test)').add_to(m)
    folium.PolyLine(coords_test_pred, color='#27ae60', weight=2, opacity=0.85,
                    dash_array='6 6', tooltip='Trayectoria predicha').add_to(m)

    grupo_real_mig.add_to(m); grupo_real_est.add_to(m)
    grupo_pred.add_to(m); grupo_lineas.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    # Resumen flotante
    err_med  = pred['error_km'].median()
    acc_top1 = pred['acierto'].mean()
    n_mig    = (pred['estado_hmm'] == 0).sum()
    titulo = (f"<div style='position:fixed;top:10px;left:50px;z-index:9999;"
              f"background:white;padding:8px 12px;border:1px solid #888;"
              f"font-family:sans-serif;font-size:13px;'>"
              f"<b>Animal {animal_id}</b><br>"
              f"Test: {len(pred)} días ({n_mig} migración) — "
              f"Top-1 {acc_top1:.1%} — mediana error {err_med:.1f} km"
              f"</div>")
    m.get_root().html.add_child(folium.Element(titulo))

    if output_html:
        m.save(output_html)
    return m

In [11]:
# Animales candidatos: los que más test tienen y al menos algo de migración (si hay)
stats_animal = test_df.groupby('animal_id').agg(
    n_test=('date', 'size'),
    n_mig=('estado_hmm', lambda s: (s == 0).sum()),
    err_med=('error_km', 'median'),
    acc=('acierto', 'mean'),
).sort_values(['n_mig', 'n_test'], ascending=False)
stats_animal.head(10)

,n_test,n_mig,err_med,acc
animal_id,,,,
91916A,379,35,22.831146,0.905013
91732A,78,27,30.972402,0.564103
91803A,22,22,37.857239,0.363636
91779A,19,19,93.770903,0.105263
91823A,259,16,19.186960,0.953668
91832A,161,15,36.873668,0.366460
91738A,17,14,83.935188,0.000000
91762A,36,13,41.860267,0.361111
91764A,22,13,29.372002,0.454545


In [12]:
# Generar mapa para los 3 animales con más migración en test
candidatos = stats_animal.head(3).index.tolist()
print('Animales seleccionados:', candidatos)

for aid in candidatos:
    out = f'../img/o5/mapa_{aid}.html'
    mapa_animal(aid, df_filt, test_df, output_html=out)
    print(f'  → {out}')

Animales seleccionados: ['91916A', '91732A', '91803A']


  → ../img/o5/mapa_91916A.html
  → ../img/o5/mapa_91732A.html


  → ../img/o5/mapa_91803A.html


In [13]:
# Vista previa del primer mapa dentro del notebook
mapa_animal(candidatos[0], df_filt, test_df)

## 6. Mapa global del error

Heatmap-like sobre todos los aciertos/fallos del test set: cada punto coloreado por error en km. Útil para ver en qué zona geográfica el modelo falla más.

In [14]:
from folium.plugins import HeatMap

centro = [test_df['next_lat'].mean(), test_df['next_lon'].mean()]
m_global = folium.Map(location=centro, zoom_start=4, tiles='cartodbpositron')

# Heatmap ponderado por error_km (cap a 300 km para evitar dominancia de outliers)
data_heat = [[r['next_lat'], r['next_lon'], min(r['error_km'], 300)] for _, r in test_df.iterrows()]
HeatMap(data_heat, radius=12, blur=18, min_opacity=0.3).add_to(m_global)

# Marcadores aciertos vs fallos por estado
for _, r in test_df.sample(min(800, len(test_df)), random_state=42).iterrows():
    color = '#2ecc71' if r['acierto'] else ('#e74c3c' if r['estado_hmm'] == 0 else '#3498db')
    folium.CircleMarker([r['next_lat'], r['next_lon']], radius=2,
                        color=color, fill=True, fill_opacity=0.6,
                        tooltip=f"{r['animal_id']} {r['date'].strftime('%Y-%m-%d')} — {r['error_km']:.0f} km"
                       ).add_to(m_global)

m_global.save('../img/o5/mapa_global_error.html')
print('Mapa global guardado en img/o5/mapa_global_error.html')
m_global

Mapa global guardado en img/o5/mapa_global_error.html


## 7. Conclusiones del módulo de visualización

Las cifras concretas se actualizan tras cada ejecución. La interpretación general:

- En **estado estacionario** la predicción cae casi siempre en la misma celda que la real → la línea predicha (verde) se solapa con la real (negra). El error mediano es del orden de **pocos km** (rejilla 0,5° ≈ 35–55 km de lado).
- En **estado de migración** la línea verde se separa de la negra; los errores diarios pueden alcanzar varios cientos de km. Esto refleja la dificultad estructural identificada en O4: sin viento, el modelo no tiene señal sobre la dirección del salto migratorio.
- El mapa global heat-map muestra que los **fallos grandes** se concentran en los corredores migratorios (golfo de Vizcaya, costa atlántica de Marruecos, Sahel) más que en las zonas de invernada.